# 02 | WorldCereal 2026: reference and spatial embeddings
**Ibrahim Kabore · ESA LTC 2025 extension**

The historical 2025 checkpoint is unavailable. We preserve Notebook 01, use the official August 2026 global seasonal model, and recompute both reference and image embeddings with one encoder. This is an independent experiment, not an exact reproduction of the original training. No neural weights are updated. The custom classifier will be fitted in Notebook 03.

**Sources:** [WorldCereal default model and presets](https://github.com/WorldCereal/worldcereal-classification/blob/bf8667902f583ec71fcca6b1a7bd1e48f64000f7/src/worldcereal/openeo/parameters.py), [seasonal pooling](https://github.com/WorldCereal/worldcereal-classification/blob/bf8667902f583ec71fcca6b1a7bd1e48f64000f7/src/worldcereal/train/data.py), [PromethEO](https://github.com/WorldCereal/prometheo/tree/v0.1.6). Run sequentially on A100. Stop at any failed assertion; share error rather than bypass it.

## 1. Environment
Pin the official source revision and install the matching PromethEO dependency through WorldCereal. A new Colab GPU runtime is recommended because the earlier notebook installed PromethEO 0.0.3. Restart the session if Colab requests it after installation, then resume from the imports.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','worldcereal @ git+https://github.com/WorldCereal/worldcereal-classification.git@bf8667902f583ec71fcca6b1a7bd1e48f64000f7'])

## 2. Project paths and imports
Use the existing Drive project. New outputs and models are isolated in `2026` folders. The saved 2025 experiment manifest supplies the reference-group split.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/ESA_EO_Projects')
matches=[p.parent.parent for p in root.rglob('extractions.parquet') if (p.parent/'processed_training_df.parquet').is_file()]
assert len(matches)==1, matches
PROJECT=matches[0]; DATA=PROJECT/'data'; BASE=PROJECT/'outputs'; OLD=PROJECT/'models'
OUT=BASE/'2026'; MODELS=OLD/'2026'
for d in [OUT,MODELS]: d.mkdir(parents=True,exist_ok=True)
print(PROJECT)
import numpy as np, pandas as pd, xarray as xr, torch, requests, hashlib, zipfile, io
from importlib.metadata import version
from pyproj import CRS, Transformer
from tqdm.auto import tqdm
from prometheo.models import Presto
from prometheo.models.pooling import PoolingMethods
from prometheo.predictors import Predictors,S1_BANDS,S2_BANDS,METEO_BANDS,DEM_BANDS,NODATAVALUE
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
from worldcereal.utils.models import load_model_artifact
from worldcereal.local_inference import run_seasonal_inference
assert torch.cuda.is_available(), 'Select GPU runtime'
print(torch.cuda.get_device_name(0),torch.__version__,version('prometheo'),version('worldcereal'))
old_manifest=json.loads((OLD/'experiment_manifest.json').read_text())
FEATURES=[f'presto_ft_{i}' for i in range(128)]

## 3. Download and verify the encoder archive
The confirmed HTTP 200 source contains a seasonal model bundle rather than a standalone checkpoint. Read `config.json` to identify its encoder-only member, verify the declared fingerprint, and load tensor weights with strict architecture matching. We retain the archive in Drive for future sessions. Never substitute a different checkpoint silently.

In [ ]:
MODEL_URL=DEFAULT_SEASONAL_MODEL_URL
assert '202608211601.zip' in MODEL_URL, MODEL_URL
bundle=MODELS/'global_seasonal_20260821.zip'
if not bundle.exists():
    r=requests.get(MODEL_URL,timeout=(30,120)); r.raise_for_status()
    assert zipfile.is_zipfile(io.BytesIO(r.content)), 'Not a valid model ZIP'
    bundle.write_bytes(r.content)
artifact=load_model_artifact(bundle,cache_root=Path('/content/wc_cache'),encoder_only=True)
config=artifact.manifest
print(json.dumps(config,indent=2)[:12000])
rel=config['artifacts']['checkpoints']['encoder_only']
encoder_path=artifact.extract_dir/rel
assert encoder_path.is_file(), f'Encoder is not included: {rel}'
fingerprint=hashlib.sha256(encoder_path.read_bytes()).hexdigest()
declared=(config.get('backbone') or {}).get('fingerprint')
if declared: assert fingerprint==declared
encoder=Presto()
encoder.load_state_dict(torch.load(encoder_path,map_location='cpu',weights_only=True),strict=True)
encoder=encoder.to('cuda').eval()
print('Encoder verified:',fingerprint)

## 4. Define a common season and prepare data
We use January–December of the label year for the encoder context and pool April–August (five months) for classification. This exploratory season includes June, when nearly every reference label is valid. It is the same calendar definition for every reference year and the 2024 cube. Only complete 12-slot reference windows are retained; optical gaps within those months remain masked. This avoids inventing temporal slots and makes the comparison honest. Following the current official inference default, B8A is masked for both datasets.

In [ ]:
ref=pd.read_parquet(DATA/'processed_training_df.parquet')
raw=pd.read_parquet(DATA/'extractions.parquet')
ds=xr.open_dataset(DATA/'worldcereal-preprocessedinputs-french-demo.nc')
BAND_MAP={**{f'S2-L2A-{b}':b.replace('B0','B') for b in ['B02','B03','B04','B05','B06','B07','B08','B8A','B11','B12']},'S1-SIGMA0-VV':'VV','S1-SIGMA0-VH':'VH','AGERA5-PRECIP':'precipitation','AGERA5-TMEAN':'temperature','elevation':'elevation','slope':'slope'}
assert set(BAND_MAP)<=set(raw.columns) and set(BAND_MAP)<=set(ds.data_vars)
POOL_MONTHS=[4,5,6,7,8]; MAP_YEAR=2024; BATCH=512
MASK_B8A=True
raw['month_start']=pd.to_datetime(raw.timestamp,utc=True).dt.tz_localize(None).dt.to_period('M').dt.to_timestamp()
ref['label_year']=pd.to_datetime(ref.valid_time).dt.year
assert not ref.sample_id.duplicated().any()
print('Reference samples:',len(ref),'| Raw rows:',len(raw),'| Cube dimensions:',dict(ds.sizes))

## 5. Shared preprocessing and seasonal pooling
Both reference and spatial batches are passed through this exact function. It performs the scaling present in WorldCereal's predictor builder (encoded radar to dB, precipitation to meters, temperature to Kelvin), masks missing data, converts geographic coordinates and passes monthly timestamps. The network returns 12 vectors per pixel. Average the five seasonal vectors to produce a 128-dimensional representation. This is custom feature extraction using official model inputs, not the complete official classifier.

In [ ]:
def encode(values,lon,lat,dates):
    n=len(lon); t=len(dates); assert t==12
    s1=np.full((n,1,1,t,len(S1_BANDS)),NODATAVALUE,np.float32)
    s2=np.full((n,1,1,t,len(S2_BANDS)),NODATAVALUE,np.float32)
    meteo=np.full((n,1,1,t,len(METEO_BANDS)),NODATAVALUE,np.float32)
    dem=np.full((n,1,1,len(DEM_BANDS)),NODATAVALUE,np.float32)
    for source,band in BAND_MAP.items():
        a=np.asarray(values[source],dtype=np.float32).copy(); assert a.shape==(n,t)
        ok=np.isfinite(a)&(a!=NODATAVALUE); a[~ok]=NODATAVALUE
        if band in S2_BANDS:
            if band=='B8A' and MASK_B8A: a[:]=NODATAVALUE
            s2[:,0,0,:,S2_BANDS.index(band)]=a
        elif band in S1_BANDS:
            ok &= a>0; a[~ok]=NODATAVALUE; a[ok]=20*np.log10(a[ok])-83
            s1[:,0,0,:,S1_BANDS.index(band)]=a
        elif band in METEO_BANDS:
            a[ok]/=100000 if band=='precipitation' else 100
            meteo[:,0,0,:,METEO_BANDS.index(band)]=a
        elif band in DEM_BANDS:
            dem[:,0,0,DEM_BANDS.index(band)]=a[:,0]
        else: raise ValueError(band)
    dates=pd.DatetimeIndex(dates)
    stamps=np.stack([dates.day,dates.month,dates.year],axis=1)
    predictor=Predictors(s1=s1,s2=s2,meteo=meteo,dem=dem,latlon=np.stack([lat,lon],axis=1).astype('float32').reshape(n,1,1,2),timestamps=np.broadcast_to(stamps,(n,t,3)).copy())
    with torch.inference_mode():
        z=encoder(predictor,eval_pooling=PoolingMethods.TIME).reshape(n,t,128)
        keep=np.flatnonzero(dates.month.isin(POOL_MONTHS)).tolist()
        z=z[:,keep,:].mean(dim=1).cpu().numpy()
    return z
def observed(v):
    usable=np.zeros(next(iter(v.values())).shape,dtype=bool)
    for b,a in v.items():
        if b.startswith(('S1-','S2-')) and not (MASK_B8A and b.endswith('B8A')):
            usable|=np.isfinite(a)&(a!=NODATAVALUE)&((a>0) if b.startswith('S1-') else True)
    return usable.sum(axis=1)>=6

## 6. Align reference samples without leakage
Join original sample IDs and reference sources, then pivot the monthly observations into ordered sequences. If a sample contains duplicate monthly rows, use a median of valid values. Keep only samples with 12 actual calendar slots and at least six observable months. Record retained and excluded samples; the original baseline will be re-evaluated on the same retained test samples if available.

In [ ]:
keys=['ref_id','sample_id']; bands=list(BAND_MAP)
raw=raw.merge(ref[keys+['label_year']],on=keys,how='inner',validate='many_to_one')
raw=raw[raw.month_start.dt.year==raw.label_year].copy()
raw['month']=raw.month_start.dt.month
raw[bands]=raw[bands].replace([NODATAVALUE,np.inf,-np.inf],np.nan)
g=raw.groupby(keys+['month'],sort=False)
months=g.size().unstack('month',fill_value=0).reindex(columns=range(1,13),fill_value=0)
complete=months.index[(months>0).all(axis=1)]
wide=g[bands].median().unstack('month').reindex(index=complete,columns=pd.MultiIndex.from_product([bands,range(1,13)]))
meta=ref.set_index(keys).loc[wide.index,['downstream_class','label_year','lat','lon']].reset_index()
vals={b:wide[b].to_numpy(dtype='float32') for b in bands}
ok=observed(vals)&np.isfinite(meta.lat)&np.isfinite(meta.lon)
meta=meta.loc[ok].reset_index(drop=True); vals={b:a[np.asarray(ok)] for b,a in vals.items()}
meta['split']=np.where(meta.ref_id.isin(old_manifest['train_sources']),'train',np.where(meta.ref_id.isin(old_manifest['test_sources']),'test','unknown'))
assert not (meta.split=='unknown').any()
print('Retained:',len(meta),'Excluded:',len(ref)-len(meta)); display(pd.crosstab(meta.downstream_class,meta.split))
assert meta.downstream_class.nunique()==8 and (pd.crosstab(meta.downstream_class,meta.split)>0).all().all()

## 7. GPU smoke test and full reference encoding
First encode eight samples from one label year to confirm correct shape, finite values and GPU execution. Compare repeated inference to ensure deterministic evaluation. Then process batches grouped by year (each has its own dates), preserving row order. Save reference vectors only after successful completion.

In [ ]:
yr=int(meta.label_year.iloc[0]); ix=np.flatnonzero(meta.label_year.to_numpy()==yr)[:8]
dates=pd.date_range(f'{yr}-01-01',periods=12,freq='MS')
args=({b:a[ix] for b,a in vals.items()},meta.lon.to_numpy()[ix],meta.lat.to_numpy()[ix],dates)
probe=encode(*args); again=encode(*args)
assert probe.shape==(len(ix),128) and np.isfinite(probe).all() and np.allclose(probe,again,atol=1e-5)
print('Probe:',probe.shape,'| GPU allocated MB:',torch.cuda.memory_allocated()/1e6)

### 7.1 Encode all retained reference samples
This is a forward pass, not fine-tuning. Eight crop categories are preserved and the split is assigned using the original manifest. Save the dataset and provenance before proceeding to the map.

In [ ]:
Z=np.empty((len(meta),128),np.float32)
for yr in sorted(meta.label_year.unique()):
    ids=np.flatnonzero(meta.label_year.to_numpy()==yr); dates=pd.date_range(f'{yr}-01-01',periods=12,freq='MS')
    for k in tqdm(range(0,len(ids),BATCH),desc=str(yr)):
        ix=ids[k:k+BATCH]; Z[ix]=encode({b:a[ix] for b,a in vals.items()},meta.lon.to_numpy()[ix],meta.lat.to_numpy()[ix],dates)
assert np.isfinite(Z).all()
new=pd.concat([meta,pd.DataFrame(Z,columns=FEATURES)],axis=1)
new.to_parquet(OUT/'reference_embeddings.parquet',index=False)
provenance={'checkpoint_sha256':fingerprint,'model_url':MODEL_URL,'pool_months':POOL_MONTHS,'context':'Jan-Dec','mask_b8a':MASK_B8A,'features':FEATURES,'map_year':MAP_YEAR,'retained':len(new),'original':len(ref),'prometheo':version('prometheo')}
(MODELS/'embedding_manifest.json').write_text(json.dumps(provenance,indent=2))
print('Saved',new.shape)

## 8. Generate spatial embeddings
For the French 2024 cube, create the same 12-month input and same April–August pooled embedding. Use batches of 512 pixels and georeference each result with the original UTM grid. Non-observed pixels are NaN. Start with a small non-square tile to catch reshaping errors.

In [ ]:
crs=CRS.from_wkt(ds.crs.attrs['crs_wkt']); assert crs.to_epsg()==32631
season=ds.sel(t=slice(f'{MAP_YEAR}-01-01',f'{MAP_YEAR}-12-31')); dates=pd.DatetimeIndex(season.t.values)
assert dates.equals(pd.date_range(f'{MAP_YEAR}-01-01',periods=12,freq='MS'))
transformer=Transformer.from_crs(crs,4326,always_xy=True)
def tile_embed(tile):
    yy,xx=np.meshgrid(tile.y.values,tile.x.values,indexing='ij'); lon,lat=transformer.transform(xx,yy)
    v={b:tile[b].transpose('y','x','t').values.reshape(-1,12) for b in bands}
    e=np.full((lon.size,128),np.nan,np.float32); ids=np.flatnonzero(observed(v))
    for k in range(0,len(ids),BATCH):
        ix=ids[k:k+BATCH]; e[ix]=encode({b:a[ix] for b,a in v.items()},lon.ravel()[ix],lat.ravel()[ix],dates)
    return e.reshape(len(tile.y),len(tile.x),128)
pilot=tile_embed(season.isel(y=slice(0,20),x=slice(0,30)))
assert pilot.shape==(20,30,128) and np.isfinite(pilot).all(axis=-1).any()
print('Spatial pilot:',pilot.shape)

### 8.1 Full image feature extraction
Export the complete embedding raster with X/Y coordinates and provenance. This is not yet a crop classification map.

In [ ]:
import rioxarray
h,w=season.sizes['y'],season.sizes['x']; E=np.full((h,w,128),np.nan,np.float32)
for y0 in tqdm(range(0,h,50),desc='Image tiles'):
    for x0 in range(0,w,50):
        z=tile_embed(season.isel(y=slice(y0,y0+50),x=slice(x0,x0+50)))
        E[y0:y0+z.shape[0],x0:x0+z.shape[1]]=z
da=xr.DataArray(E.transpose(2,0,1),dims=['feature','y','x'],coords={'feature':FEATURES,'y':season.y,'x':season.x},name='presto_embedding').rio.write_crs(crs)
da.attrs['checkpoint_sha256']=fingerprint; da.attrs['pool_months']='4,5,6,7,8'; da.attrs['context_year']=MAP_YEAR
da.to_netcdf(OUT/'spatial_embeddings_2024.nc',engine='netcdf4',encoding={'presto_embedding':{'zlib':True,'complevel':2}})
print('Valid pixels:',np.isfinite(E).all(axis=-1).sum(),'/',h*w)

## 9. Official cropland mask
Our eight-class Random Forest cannot recognize non-cropland. Use the separate land-cover head in the official global bundle to derive a cropland mask. The official engine manages its own seasonal pooling; the mask is a separate product, not ground truth. Only pixels classified as cropland by this head will receive our crop predictions. Validate the returned grid and class codes.

In [ ]:
mask_result=run_seasonal_inference(season,seasonal_model_zip=bundle,season_ids=['tc-annual'],season_windows={'tc-annual':(f'{MAP_YEAR}-01-01',f'{MAP_YEAR}-12-31')},enable_cropland_head=True,enable_croptype_head=False,mask_cropland=False,batch_size=BATCH,device='cuda',mask_b8a=MASK_B8A,cache_root=Path('/content/wc_cache'))
print(mask_result)
mask=mask_result['cropland_classification'].transpose('y','x')
assert np.array_equal(mask.x,season.x) and np.array_equal(mask.y,season.y)
assert set(np.unique(mask.values)).issubset({0,1,255})
mask=mask.rio.write_crs(crs).rio.write_nodata(255)
mask.rio.to_raster(OUT/'cropland_mask_2024.tif')
print('Cropland pixels:',int((mask.values==1).sum()))

## Checkpoint
Send the model fingerprint, retained sample counts, GPU pilot result and spatial mask count. If Section 9 fails, the embedding outputs are still valid; do not manufacture a cropland mask. Notebook 03 can run from the reference embeddings independently of Section 9.